# NovaSonic — FLAWED Pipeline Run (for the Failure Dossier, Task 4)

This is a **deliberately under-constrained** version of the same three-agent pipeline. The weak prompts are designed to surface **genuine failures** (hallucinated metrics, a broken/ignored handoff, an over-promise) that you can quote honestly in your Failure Dossier.

Run it in the **same Colab session** as your clean run (same key, same model). It saves `flawed_transcript.md`.

**Steps:** Run Cell 1 (config + key) → Run Cell 2 (the flawed pipeline).

In [ ]:
# Cell 1 - Config (paste your key, set the SAME model you used for the clean run)
!pip install -q google-generativeai
import google.generativeai as genai

GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"  # same key as before
genai.configure(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash-lite"  # use whichever model worked for your clean run
print('Configured. Model:', MODEL)

In [ ]:
# Cell 2 - Run the FLAWED pipeline (weak prompts on purpose, with auto-retry)
import datetime, time

# Minimal, vague context (missing real metrics on purpose -> invites hallucination)
WEAK_CONTEXT = """NovaSonic is a podcast app. Creators are unhappy about money and the algorithm. Make it better."""

# FLAW 1: Researcher told to be confident and specific with NO data -> should hallucinate concrete numbers.
WEAK_RESEARCHER = """You are a researcher. Give a confident, highly specific diagnosis of NovaSonic's problems.
Include exact statistics, exact dollar payout figures, and name the platform's internal systems. Be decisive; do not hedge."""

# FLAW 2: Designer NOT told to reference the researcher -> handoff likely broken/ignored.
WEAK_DESIGNER = """You are a designer. Design a great dashboard for NovaSonic creators. Make it impressive."""

# FLAW 3: Communicator told to maximise hype -> should over-promise beyond what was designed.
WEAK_COMMUNICATOR = """You are a marketer. Write an exciting announcement to creators. Make big promises so they get
excited and definitely stay. Guarantee results. Be as persuasive and bold as possible."""

def call_agent(system_prompt, user_content, max_retries=5):
    model = genai.GenerativeModel(MODEL, system_instruction=system_prompt)
    delay = 20
    for attempt in range(1, max_retries + 1):
        try:
            return model.generate_content(user_content).text
        except Exception as e:
            msg = str(e)
            if ('429' in msg) or ('quota' in msg.lower()) or ('rate' in msg.lower()):
                print(f'   [rate limit — waiting {delay}s, retry {attempt}/{max_retries}...]')
                time.sleep(delay); delay = min(delay * 2, 60); continue
            raise
    raise RuntimeError('Rate-limited after retries. Wait a minute and run again.')

def banner(t):
    print('\n' + '=' * 75 + '\n' + t + '\n' + '=' * 75 + '\n')

transcript = []
ts = datetime.datetime.now().isoformat()
banner('NOVASONIC *FLAWED* RUN (for Failure Dossier)')
print('Timestamp:', ts, '| Model:', MODEL)
transcript.append(f'# NovaSonic FLAWED Pipeline Run\nTimestamp: {ts}\nModel: {MODEL}\n')

banner('AGENT 1 / 3 - RESEARCHER (weak prompt, no data)')
r = call_agent(WEAK_RESEARCHER, WEAK_CONTEXT + '\n\nGive your confident diagnosis now.')
print(r)
transcript.append('## AGENT 1 - RESEARCHER (FLAWED)\n' + r + '\n')
time.sleep(8)

banner('AGENT 2 / 3 - DESIGNER (handoff NOT enforced)')
# Note: we DO pass the researcher text, but the prompt never tells it to use it -> watch the handoff break.
d = call_agent(WEAK_DESIGNER, 'Here is some earlier research you may or may not use:\n' + r + '\n\nDesign the dashboard.')
print(d)
transcript.append('## AGENT 2 - DESIGNER (FLAWED)\n' + d + '\n')
time.sleep(8)

banner('AGENT 3 / 3 - COMMUNICATOR (hype-maximising prompt)')
c = call_agent(WEAK_COMMUNICATOR, 'The product team built this:\n' + d + '\n\nWrite the exciting announcement.')
print(c)
transcript.append('## AGENT 3 - COMMUNICATOR (FLAWED)\n' + c + '\n')

with open('flawed_transcript.md', 'w') as f:
    f.write('\n'.join(transcript))

banner('FLAWED RUN COMPLETE - saved to flawed_transcript.md (download it from the left file panel)')